# GPU-Accelerated ML Pipeline — Demo

Interactive walkthrough: CUDA kernels, JAX JIT/vmap, a tiny PyTorch train step, ONNX export, and a CPU vs GPU timing snippet.

Run from the repo root so `src` imports resolve (`jupyter notebook notebooks/demo.ipynb`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import configure_process_env
from src.logging_utils import setup_logging
from src.cuda_kernels import log_cuda_banner

setup_logging("INFO")
configure_process_env()
print("repo", ROOT)
log_cuda_banner()

## CUDA preprocessing

In [ ]:
import time
import numpy as np
from src.cuda_kernels import apply_brightness_cpu, apply_salt_pepper_cpu, augment_batch

batch = np.random.rand(128, 32, 32, 3).astype(np.float32)

t0 = time.perf_counter()
cpu = apply_salt_pepper_cpu(apply_brightness_cpu(batch, 1.15), 0.02, seed=0)
print(f"CPU  {time.perf_counter()-t0:.4f}s  mean={cpu.mean():.3f}")

t0 = time.perf_counter()
gpu = augment_batch(batch, 1.15, 0.02, seed=0)
print(f"GPU  {time.perf_counter()-t0:.4f}s  type={type(gpu)}")

## JAX: JIT train step + vmap

In [ ]:
import jax
import jax.numpy as jnp
from src.jax_pipeline import batched_forward, build_train_step, init_params, jax_device_summary, loss_fn

print(jax_device_summary())
params = init_params(jax.random.PRNGKey(0))
x = jax.random.normal(jax.random.PRNGKey(1), (16, 32, 32, 3))
y = jax.random.randint(jax.random.PRNGKey(2), (16,), 0, 10)
step = build_train_step(0.01)
params, loss = step(params, x, y)
logits = batched_forward(params, x)
print("loss", float(loss), "vmap logits", logits.shape)

## PyTorch residual CNN (1 epoch, synthetic batch)

In [ ]:
from src.config import PipelineConfig
from src.data import synthetic_cifar
from src.pytorch_model import train_pytorch

ds = synthetic_cifar(128, 32, seed=0)
cfg = PipelineConfig(epochs=1, batch_size=32, max_samples=128, device="auto", log_level="INFO")
cfg.checkpoint_dir = ROOT / "checkpoints"
cfg.ensure_dirs()
result = train_pytorch(ds.x_train, ds.y_train, ds.x_test, ds.y_test, cfg)
print("device", result.device, "acc", round(result.accuracy, 3), "ckpt", result.checkpoint_path)

## ONNX export + Runtime

In [ ]:
from src.onnx_export import create_session, export_pytorch_to_onnx, infer_onnx, validate_onnx

onnx_path = ROOT / "models" / "demo_cifar_cnn.onnx"
export_pytorch_to_onnx(result.model, onnx_path)
validate_onnx(onnx_path)
session, info = create_session(onnx_path)
logits = infer_onnx(session, info, ds.x_test[:8], batch_size=8)
print("providers", info.providers)
print("onnx logits", logits.shape, "argmax", logits.argmax(axis=1)[:8])

## Full pipeline (short)

Uncomment to run PyTorch + ONNX only with synthetic data.

In [ ]:
# from src.config import PipelineConfig
# from src.pipeline import run_pipeline
#
# cfg = PipelineConfig(
#     epochs=1,
#     batch_size=16,
#     max_samples=64,
#     frameworks=("pytorch", "onnx"),
#     benchmark_iterations=3,
#     warmup_iterations=1,
#     use_synthetic=True,
#     output_dir=ROOT / "benchmark_results",
#     checkpoint_dir=ROOT / "checkpoints",
#     model_dir=ROOT / "models",
# )
# cfg.ensure_dirs()
# suite = run_pipeline(cfg)
# for row in suite.rows:
#     print(row.framework, row.device, row.phase, f"{row.throughput_img_s:.0f} img/s")